In [ ]:
# ==============================================
# EMPLOYEES ATTRITION DATASETS - USING PIPELINE
# ==============================================

# =============================
# **** IMPORT LIBRARIES ****
# =============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =========================
# **** Load data ****
# =========================
df = pd.read_csv(r"C:\Users\priya\Downloads\employee_attrition_dataset.csv")
df = df.drop('Employee_ID', axis=1)

# =========================
# Prepare target 
# =========================
le = LabelEncoder()
df['Attrition'] = le.fit_transform(df['Attrition'])  # No=0, Yes=1

# =========================
# Split features and target
# =========================
X = df.drop('Attrition', axis=1)
y = df['Attrition']

# =========================
# Identify columns
# =========================
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# =========================
# Create preprocessor
# =========================
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

# =========================
# Create pipeline
# =========================
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# =========================
# Train-test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================
# Train
# =========================
pipeline.fit(X_train, y_train)

# =========================
# Evaluate
# =========================
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay', 'Leave']))

# ==============================
# Feature importance (top 10)
# ==============================
importances = pipeline.named_steps['classifier'].feature_importances_
features = (list(numeric_cols) + 
           list(pipeline.named_steps['preprocessor']
                .named_transformers_['cat']
                .get_feature_names_out(categorical_cols)))
imp_df = pd.DataFrame({'feature': features, 'importance': importances})
print("\nTop 10 Important Features:")
print(imp_df.sort_values('importance', ascending=False).head(10))

# ============================
# Quick prediction function
# ============================
def predict_attrition(new_data):
    """new_data should be DataFrame with same columns"""
    pred = pipeline.predict(new_data)[0]
    prob = pipeline.predict_proba(new_data)[0]
    return f"Will Leave: {'Yes' if pred==1 else 'No'} (Probability: {prob[1]:.2%})"

C:\Users\priya\AppData\Local\Temp\ipykernel_14136\1098436529.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


Accuracy: 0.8450 (84.50%)

Classification Report:
              precision    recall  f1-score   support

        Stay       0.84      1.00      0.92       169
       Leave       0.00      0.00      0.00        31

    accuracy                           0.84       200
   macro avg       0.42      0.50      0.46       200
weighted avg       0.71      0.84      0.77       200


Top 10 Important Features:
                          feature  importance
2                  Monthly_Income    0.073522
3                     Hourly_Rate    0.071336
10       Training_Hours_Last_Year    0.063202
12  Average_Hours_Worked_Per_Week    0.060746
0                             Age    0.060055
17             Distance_From_Home    0.059477
4                Years_at_Company    0.056255
13                    Absenteeism    0.053821
5           Years_in_Current_Role    0.050149
6      Years_Since_Last_Promotion    0.044295


C:\Users\priya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\priya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\priya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_clas